# StreamLoop — Ajuste sistemático del modelo de cancelación

Objetivo: maximizar el **recall de Churn=Yes**, porque para StreamLoop es más costoso no detectar a un cliente que cancela que contactar a un cliente que finalmente no cancela. El test se reserva y solo se usa para la línea base y la evaluación final.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import loguniform
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(URL)
df = df.drop(columns=['customerID'], errors='ignore')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df = df.dropna(subset=['Churn']).copy()
X = df.drop(columns='Churn')
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f'Dimensiones: {df.shape}; train={X_train.shape}, test={X_test.shape}')

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])
pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])

def metrics(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {'accuracy': accuracy_score(y_eval, pred), 'precision': precision_score(y_eval, pred, zero_division=0),
            'recall': recall_score(y_eval, pred), 'f1': f1_score(y_eval, pred)}

baseline = pipeline.fit(X_train, y_train)
baseline_metrics = metrics(baseline, X_test, y_test)
print('Baseline (test, una sola evaluación):', baseline_metrics)
print(classification_report(y_test, baseline.predict(X_test), target_names=['No churn', 'Churn']))

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
param_distributions = {
    'classifier__C': loguniform(1e-3, 1e2),
    'classifier__solver': ['liblinear', 'lbfgs'],
    'classifier__class_weight': [None, 'balanced'],
}
random_search = RandomizedSearchCV(
    pipeline, param_distributions=param_distributions, n_iter=20,
    scoring='recall', cv=cv, n_jobs=-1, refit=True,
    random_state=RANDOM_STATE, return_train_score=True
)
random_search.fit(X_train, y_train)
random_results = pd.DataFrame(random_search.cv_results_).sort_values('rank_test_score')
print('RandomizedSearchCV — mejores candidatos:')
display(random_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head(10))

In [ ]:
best_random = random_search.best_params_
best_c = best_random['classifier__C']
grid_c = sorted(set([max(1e-3, best_c / 3), best_c, best_c * 3]))
grid = {
    'classifier__C': grid_c,
    'classifier__solver': [best_random['classifier__solver']],
    'classifier__class_weight': [best_random['classifier__class_weight']],
}
grid_search = GridSearchCV(
    pipeline, param_grid=grid, scoring='recall', cv=cv, n_jobs=-1,
    refit=True, return_train_score=True
)
grid_search.fit(X_train, y_train)
grid_results = pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score')
print('GridSearchCV — todos los candidatos:')
display(grid_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']])

In [ ]:
stability = grid_results[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].copy()
stability['lower_estimate'] = stability['mean_test_score'] - stability['std_test_score']
stability['upper_estimate'] = stability['mean_test_score'] + stability['std_test_score']
print('Análisis de estabilidad (media +/- desviación estándar):')
display(stability)
print('Mejores hiperparámetros:', grid_search.best_params_)
print('Mejor recall CV:', grid_search.best_score_)

# Evaluación final: el test se usa aquí por segunda y última vez.
final_model = grid_search.best_estimator_
final_metrics = metrics(final_model, X_test, y_test)
print('Final (test, segunda y última evaluación):', final_metrics)
print('Matriz de confusión final:\n', confusion_matrix(y_test, final_model.predict(X_test)))
print(classification_report(y_test, final_model.predict(X_test), target_names=['No churn', 'Churn']))